# 02 - Dataset Preparation (Google Colab)
Load raw dataset, validate JSON structure, split into train/validation, and preview tokenized samples.

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
COLAB_ROOT = '/content/drive/MyDrive/colab'
os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import yaml

In [ ]:
# Load configuration
with open("configs/training_config.yaml", "r") as f:
    config = yaml.safe_load(f)

RAW_PATH = config["raw_dataset"]
TRAIN_PATH = config["train_dataset"]
VAL_PATH = config["validation_dataset"]
TRAIN_SPLIT = config["train_split"]
SEED = config["seed"]

print(f"Raw dataset: {RAW_PATH}")
print(f"Train split: {TRAIN_SPLIT}")
print(f"Seed: {SEED}")

In [ ]:
# Step 1: Load raw dataset
print("Loading raw dataset...")
with open(RAW_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} samples")
print(f"\nSample keys: {list(raw_data[0].keys())}")
print(f"\nInstruction preview: {raw_data[0]['instruction'][:100]}...")
print(f"\nInput preview: {raw_data[0]['input'][:200]}...")

In [ ]:
# Step 2: Validate JSON structure
def validate_sample(sample):
    """Validate a single dataset sample."""
    required_keys = {"instruction", "input", "output"}
    if not required_keys.issubset(sample.keys()):
        return False, "Missing required keys"

    try:
        output = json.loads(sample["output"])
    except (json.JSONDecodeError, TypeError):
        return False, "Invalid JSON in output"

    required_output_keys = {
        "ats_score", "score_breakdown", "matched_skills",
        "missing_skills", "weak_bullets", "formatting_issues",
        "overall_feedback"
    }
    if not required_output_keys.issubset(output.keys()):
        return False, f"Missing output keys: {required_output_keys - output.keys()}"

    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False, "Invalid ATS score"

    return True, "Valid"

valid_samples = []
invalid_samples = []

for i, sample in enumerate(raw_data):
    is_valid, reason = validate_sample(sample)
    if is_valid:
        valid_samples.append(sample)
    else:
        invalid_samples.append((i, reason))

print(f"Valid samples: {len(valid_samples)}")
print(f"Invalid samples: {len(invalid_samples)}")

if invalid_samples:
    print("\nInvalid sample details:")
    for idx, reason in invalid_samples[:5]:
        print(f"  Sample {idx}: {reason}")

In [ ]:
# Step 3: Explore data distribution
scores = []
for sample in valid_samples:
    output = json.loads(sample["output"])
    scores.append(output["ats_score"])

print(f"ATS Score Distribution:")
print(f"  Min: {min(scores)}")
print(f"  Max: {max(scores)}")
print(f"  Mean: {sum(scores)/len(scores):.1f}")
print(f"  Median: {sorted(scores)[len(scores)//2]}")

# Score histogram (text-based)
buckets = Counter()
for s in scores:
    bucket = (s // 10) * 10
    buckets[bucket] += 1

print("\nScore Distribution:")
for bucket in sorted(buckets.keys()):
    bar = "#" * buckets[bucket]
    print(f"  {bucket:3d}-{bucket+9:3d}: {bar} ({buckets[bucket]})")

In [ ]:
# Step 4: Format prompts
def format_prompt(sample, eos_token="<|endoftext|>"):
    """Format sample into instruction-tuning prompt."""
    return (
        f"### Instruction:\n{sample['instruction']}\n\n"
        f"### Input:\n{sample['input']}\n\n"
        f"### Response:\n{sample['output']}{eos_token}"
    )

for sample in valid_samples:
    sample["text"] = format_prompt(sample)

# Check prompt lengths
lengths = [len(s["text"]) for s in valid_samples]
print(f"Prompt character lengths:")
print(f"  Min: {min(lengths)}")
print(f"  Max: {max(lengths)}")
print(f"  Mean: {sum(lengths)/len(lengths):.0f}")

print(f"\n--- Sample Formatted Prompt ---")
print(valid_samples[0]["text"][:600])
print("...")

In [ ]:
# Step 5: Split dataset
random.seed(SEED)
random.shuffle(valid_samples)

split_idx = int(len(valid_samples) * TRAIN_SPLIT)
train_data = valid_samples[:split_idx]
val_data = valid_samples[split_idx:]

print(f"Train set: {len(train_data)} samples")
print(f"Validation set: {len(val_data)} samples")

In [ ]:
# Step 6: Save processed datasets
os.makedirs(os.path.dirname(TRAIN_PATH), exist_ok=True)

with open(TRAIN_PATH, "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)

with open(VAL_PATH, "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2, ensure_ascii=False)

print(f"Saved train set to {TRAIN_PATH}")
print(f"Saved validation set to {VAL_PATH}")

In [ ]:
# Step 7: Preview tokenization
from transformers import AutoTokenizer

model_name = config["model_name"]
print(f"Loading tokenizer: {model_name}")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
except Exception:
    model_name = config["fallback_model"]
    print(f"Fallback to: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize a sample
sample_text = train_data[0]["text"]
tokens = tokenizer(sample_text, truncation=True, max_length=2048)

print(f"\nTokenizer loaded: {model_name}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Sample token count: {len(tokens['input_ids'])}")
print(f"Max seq length config: {config['max_seq_length']}")

# Token length distribution
token_lengths = []
for sample in train_data[:50]:
    toks = tokenizer(sample["text"], truncation=True, max_length=2048)
    token_lengths.append(len(toks["input_ids"]))

print(f"\nToken length stats (first 50 samples):")
print(f"  Min: {min(token_lengths)}")
print(f"  Max: {max(token_lengths)}")
print(f"  Mean: {sum(token_lengths)/len(token_lengths):.0f}")

print("\nDataset preparation complete!")